# Evaluating drift-diffusion models

After training the Selective Integration (Cao et al., 2022, eLife; https://github.com/YinanCao/multiattribute-distractor) and Mutual Inhibition (Chau et al., 2020, eLife; https://datadryad.org/dataset/doi:10.5061/dryad.k6djh9w3c) models, here we evaluate the models cross-validated predictions and overall fits.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy

In [54]:
si_trinary = pd.read_csv('selective_integration_trinary_predictions.csv', header=None, names=['target_p', 'competitor_p', 'decoy_p'])
si_binary = pd.read_csv('selective_integration_binary_predictions.csv', header=None, names=['target_p', 'competitor_p', 'decoy_p'])
mi_preds = pd.read_csv('mutual_inhibition_target_choices.csv', header=None, names=['target_p'])
raw_choices = pd.read_csv('raw_choices_for_fitting.csv')
raw_choices.loc[:, 'mi_target_p'] = mi_preds['target_p']

In [55]:
trinary_choices = raw_choices.loc[raw_choices.trinary_group==1].reset_index()
binary_choices = raw_choices.loc[raw_choices.trinary_group==0].reset_index()

In [56]:
trinary_choices.loc[:, 'SI_target_p'] = si_trinary.target_p
binary_choices.loc[:, 'SI_target_p'] = si_binary.target_p

In [57]:
trinary_choices.set_index('index', inplace=True)
binary_choices.set_index('index', inplace=True)

In [58]:
raw_choices = pd.concat([binary_choices, trinary_choices], axis=0)

In [59]:
binary_aggregated = binary_choices[['trinary_id', 'target_choice', 'mi_target_p', 'SI_target_p']].groupby('trinary_id').mean()
trinary_aggregated = trinary_choices[['trinary_id', 'target_choice', 'mi_target_p', 'SI_target_p']].groupby('trinary_id').mean()

In [60]:
decoys = trinary_aggregated - binary_aggregated

In [61]:
decoys.corr()

,target_choice,mi_target_p,SI_target_p
target_choice,1.000000,0.579593,0.739366
mi_target_p,0.579593,1.000000,0.687147
SI_target_p,0.739366,0.687147,1.000000


In [63]:
# calculate bootstrap errors for the correlation between decoy effects and predictions
def bootstrap_corr(predictions, decoy_effects, n_boot=10000):
    boot_corrs = []
    predictions = predictions.reset_index(drop=True).copy()
    decoy_effects = decoy_effects.reset_index(drop=True).copy()
    for _ in range(n_boot):
        indices = np.random.choice(range(len(predictions)), size=len(predictions), replace=True)
        boot_corr = scipy.stats.pearsonr(predictions[indices], decoy_effects[indices])[0]
        boot_corrs.append(boot_corr)
    return np.std(boot_corrs)

In [66]:
bootstrap_corr(decoys.SI_target_p, decoys.target_choice)

0.09562189384670959

## CV evaluation

In [68]:
si_cv_rmse = []
mi_cv_rmse = []
for fold_i in range(1, 26):
    # real raw choices
    raw_choices_filename = f'cv_raw_choices/fold{fold_i}_test_raw_choices.csv'
    raw_choices = pd.read_csv(raw_choices_filename)
    # mutual inhibition predictions
    mi_preds_filename = f'ddm_cv_predictions/mutual_inhibition/fold{fold_i}_preds.csv'
    mi_preds = pd.read_csv(mi_preds_filename, header=None, names=['target_p'])
    raw_choices.loc[:, 'MI_target_p'] = mi_preds['target_p']
    # selective integration predictions for binary/trinary separately
    binary_pred_filename = f'ddm_cv_predictions/selective_integration/fold{fold_i}_binary.csv'
    trinary_pred_filename = f'ddm_cv_predictions/selective_integration/fold{fold_i}_trinary.csv'
    binary_preds = pd.read_csv(binary_pred_filename, header=None, names=['target_p', 'competitor_p', 'decoy_p'])
    trinary_preds = pd.read_csv(trinary_pred_filename, header=None, names=['target_p', 'competitor_p', 'decoy_p'])
    binary_raw_choices = raw_choices.loc[raw_choices.trinary_group==0].reset_index()
    trinary_raw_choices = raw_choices.loc[raw_choices.trinary_group==1].reset_index()
    binary_raw_choices.loc[:, 'SI_target_p'] = binary_preds.target_p
    trinary_raw_choices.loc[:, 'SI_target_p'] = trinary_preds.target_p
    binary_raw_choices.set_index('index', inplace=True)
    trinary_raw_choices.set_index('index', inplace=True)
    raw_choices = pd.concat([binary_raw_choices, trinary_raw_choices], axis=0)
    # aggregate choices to calculate decoys
    binary_aggregated = binary_raw_choices[['trinary_id', 'target_choice', 'MI_target_p', 'SI_target_p']].groupby('trinary_id').mean()
    trinary_aggregated = trinary_raw_choices[['trinary_id', 'target_choice', 'MI_target_p', 'SI_target_p']].groupby('trinary_id').mean()
    test_decoys = trinary_aggregated - binary_aggregated
    si_rmse = np.sqrt(np.mean((test_decoys.target_choice - test_decoys.SI_target_p) ** 2))
    mi_rmse = np.sqrt(np.mean((test_decoys.target_choice - test_decoys.MI_target_p) ** 2))
    si_cv_rmse.append(si_rmse)
    mi_cv_rmse.append(mi_rmse)

In [69]:
print(f'''Mutual Inhibition RMSE:       {np.mean(mi_cv_rmse):.4f}
Selective Integration RMSE:   {np.mean(si_cv_rmse):.4f}''')

Mutual Inhibition RMSE:       0.0881
Selective Integration RMSE:   0.0969
